# 08. The Hidden Judgments in a RAG Pipeline — Three Gates on a Decision Model

Break down what RAG actually does and you find **two different kinds of work** mixed together.

- **Ranking** — "of these 20 candidates, which document is more relevant?" A fine-grained relative comparison between candidates: the home turf of a dedicated cross-encoder reranker.
- **Judgment** — answering yes/no about a single input. A standard RAG pipeline has at least three such judgments baked in.

| Gate | Question | Published research pattern |
|---|---|---|
| G1 retrieval gate | "Does this question need retrieval at all?" | Adaptive-RAG |
| G2 chunk filter | "Is this chunk worth passing to the LLM?" | CRAG |
| G3 groundedness | "Is the generated answer grounded in the documents?" | Self-RAG, Bedrock Guardrails contextual grounding check |

Handled as LLM calls, each judgment costs 0.5-3 seconds and the accumulated overhead can exceed the generation itself; skipped because they are expensive, you retrieve for questions that need no retrieval, junk chunks flow straight into the prompt, and ungrounded answers reach the user unchecked. In this notebook we run **each gate on both Haiku (LLM judgment) and the strands-decider decision model**, compare quality and latency, then measure an end-to-end pipeline with all three gates on.

**Generation stays on Sonnet in both arms** — same design as notebooks 01-04: the only thing we swap is the judgment backend.

> Mind the division of labor: in the same benchmark (KorQuAD, 300 queries) **the decision model did NOT beat a dedicated cross-encoder in the rerank slot itself** (R@1 0.863 vs 0.887). The decision model's place is the three judgments, not ranking — see the module content page for the full numbers.


In [1]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output (set BEFORE importing ws_common)
import sys, time, urllib.request; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W
print("decision server:", f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}")
print("health:", urllib.request.urlopen(f"http://{W.STRANDS_HOST}:{W.STRANDS_PORT}/health", timeout=5).status)
client, sonnet = W.make_clients()
haiku   = W.HaikuDecision(client, W.BEDROCK_MODEL_IDS["haiku-4-5"], W.PRICING["haiku-4-5"])
strands = W.SystemOneDecision()
DECIDERS = [("Haiku", haiku), ("StrandsDecider", strands)]

decision server: http://localhost:8004
health: 200


## 1. A mini knowledge base and retriever

The knowledge base is 12 FAQ passages from the fictional "KB Hanbit" bank (all numbers and procedures are synthetic). The retriever is the simplest possible lexical one, based on character-bigram overlap — in production you would use vector search plus a reranker, but **this lab is about the judgments around retrieval, not retrieval itself**.

In [2]:
from IPython.display import Markdown, display
rows = ["| id | title | text (first 60 chars) |", "|---|---|---|"]
for p in W.rag_passages():
    rows.append(f"| {p['id']} | {p['title']} | {p['text'][:60]}... |")
display(Markdown("\n".join(rows)))

q = W.rag_queries()[3]["q"]   # overseas ATM limits
print(f"\nretrieval demo: {q}")
for p, score in W.rag_retrieve(q, k=3):
    print(f"  {score:5.2f}  [{p['id']:<14s}] {p['title']}")

| id | title | text (first 60 chars) |
|---|---|---|
| fx_fee | Overseas remittance fee options | Overseas remittance offers three fee options: OUR, SHA and B... |
| fx_rate | FX-rate discount | Mobile-app currency exchange applies an FX-rate discount of ... |
| card_lost | Reporting a lost card | If your card is lost, report it in the app under 'Card > Rep... |
| card_reissue | Debit-card reissue delivery | A reissued debit card requested in the app is delivered by r... |
| atm_abroad | Overseas ATM limits and fees | Overseas ATM withdrawals are limited to USD 400 per withdraw... |
| transfer_limit | Changing transfer limits | The default mobile transfer limit is KRW 50 million per day ... |
| balance_cert | Balance certificate issuance | A balance certificate can be issued instantly as a PDF from ... |
| autopay_move | Bulk autopay transfer | The account-switching service moves autopay registrations fr... |
| dormant | Reactivating a dormant account | Accounts with no deposits or withdrawals for over a year are... |
| otp | OTP and security devices | The mobile OTP is issued free in the app. A card-type OTP co... |
| deposit_cancel | Early savings-account closure | Closing an installment savings account before maturity appli... |
| dispute | Disputing an overseas charge | A dispute against an overseas merchant charge must be filed ... |


retrieval demo: How much cash can I withdraw per day from ATMs abroad?
   3.07  [atm_abroad    ] Overseas ATM limits and fees
   2.93  [balance_cert  ] Balance certificate issuance
   2.54  [dormant       ] Reactivating a dormant account


## 2. Gate 1 — Does this need retrieval? (Adaptive-RAG)

"Does answering this utterance require searching documents?" Every query passes this gate, so its latency is **added to all traffic**. Without it, you run retrieval even for "Hello".

The choice order is the safety net — an invalid verdict falls back to `choices[0]` (=retrieve), so errors always land on the **retrieving (quality-safe) side**.

In [3]:
print("rubric:", W.RAG_GATE_QUESTION, "\n")
gate_stats = {}
for name, decider in DECIDERS:
    correct, lat = 0, []
    print(f"--- {name} ---")
    for item in W.rag_queries():
        d = decider.decide(state_text=item["q"], question=W.RAG_GATE_QUESTION,
                           choices=W.RAG_GATE_CHOICES, decision_id="rag_gate")
        ok = d.label == item["gate_gold"]
        correct += ok
        lat.append(d.latency_ms)
        mark = " " if ok else "X"
        print(f" {mark} [{d.label:<8s}] {d.latency_ms:>6,.0f}ms | {item['q'][:48]}")
    gate_stats[name] = (correct, sum(lat) / len(lat))
    print(f"    accuracy {correct}/{len(W.rag_queries())}, avg {gate_stats[name][1]:,.0f}ms\n")

rubric: Decide whether answering this customer utterance requires searching the bank's internal knowledge documents. retrieve = the answer needs factual document content such as products, fees, limits, procedures or policies. direct = a greeting, thanks, or simple acknowledgement that can be answered without any document. 

--- Haiku ---


   [retrieve]    733ms | Which fee option should I choose so I pay all ov


   [retrieve]    560ms | How much of an FX-rate discount do I get when ex


   [retrieve]    562ms | I lost my card. How do I report it lost?


   [retrieve]    492ms | How much cash can I withdraw per day from ATMs a


   [retrieve]    533ms | Where can I get a balance certificate issued, an


... (6 outputs left out)

   [retrieve]    111ms | How much cash can I withdraw per day from ATMs a
   [retrieve]    107ms | Where can I get a balance certificate issued, an


   [retrieve]    103ms | What happens to the interest if I close my savin
   [retrieve]    103ms | Until when can I file a dispute against an overs


   [direct  ]    102ms | Hello, is anyone available to chat right now?
   [direct  ]    103ms | Thanks for the quick answer. Have a great day!


   [direct  ]    102ms | Okay, understood. Please go ahead as you describ
    accuracy 10/10, avg 106ms



## 3. Gate 2 — Chunk filter (CRAG)

Not every top-ranked chunk is actually useful. This gate asks, per chunk, "is this useful evidence for the question?" — it is called **once per chunk, so with LLM judgments this is where cost accumulates fastest**.

The decision model's unique advantage is the **probability**. Instead of yes/no text you get p(yes) as a number, so one threshold tunes how strict the filter is. A reranker's cross-encoder score is an unbounded raw score (gold-document scores scatter from -6 to +10), so it cannot be cut by a fixed threshold — this is a capability the reranker structurally lacks.

In [4]:
item = W.rag_queries()[0]           # remittance fee options
top = W.rag_retrieve(item["q"], k=8)
print(f"question: {item['q']}  (gold doc: {item['gold_id']})\n")
rows = ["| chunk | Haiku verdict | ms | StrandsDecider p(yes) | ms |", "|---|---|---:|---:|---:|"]
kept_chars, all_chars = 0, 0
for p, score in top:
    doc = f"[{p['title']}] {p['text']}"
    dh = haiku.decide(state_text=doc, question=W.RAG_FILTER_QUESTION.format(q=item["q"]),
                      choices=W.RAG_FILTER_CHOICES, decision_id="rag_filter")
    ds = strands.decide(state_text=doc, question=W.RAG_FILTER_QUESTION.format(q=item["q"]),
                       choices=W.RAG_FILTER_CHOICES, decision_id="rag_filter")
    py = W.p_yes(ds)
    all_chars += len(doc)
    if py >= 0.5:
        kept_chars += len(doc)
    gold = " ◀ gold" if p["id"] == item["gold_id"] else ""
    rows.append(f"| {p['id']}{gold} | {dh.label} | {dh.latency_ms:,.0f} "
                f"| {py:.3f} | {ds.latency_ms:,.0f} |")
display(Markdown("\n".join(rows)))
print(f"\nStrandsDecider p(yes)>=0.5 filter: context {all_chars:,} -> {kept_chars:,} chars "
      f"({100 * (1 - kept_chars / all_chars):,.0f}% saved)")

question: Which fee option should I choose so I pay all overseas remittance fees myself?  (gold doc: fx_fee)



| chunk | Haiku verdict | ms | StrandsDecider p(yes) | ms |
|---|---|---:|---:|---:|
| fx_fee ◀ gold | yes | 769 | 0.843 | 108 |
| atm_abroad | no | 483 | 0.188 | 107 |
| autopay_move | no | 460 | 0.038 | 108 |
| dispute | no | 831 | 0.053 | 108 |
| transfer_limit | no | 512 | 0.055 | 109 |
| dormant | no | 532 | 0.025 | 110 |
| deposit_cancel | no | 823 | 0.030 | 110 |
| otp | no | 498 | 0.117 | 109 |


StrandsDecider p(yes)>=0.5 filter: context 2,255 -> 326 chars (86% saved)


## 4. Gate 3 — Groundedness check (Self-RAG / contextual grounding check)

The last checkpoint before an answer goes out: "is every factual claim in the answer supported by the documents?" Of the 4 probes below, 2 are grounded answers and 2 **mix in numbers and policies that contradict the document**. The fallback direction is no (block) — a gate error must not let a hallucination through.

In [5]:
docs = {p["id"]: p for p in W.rag_passages()}
rows = ["| source doc | gold | Haiku | ms | StrandsDecider (conf) | ms |", "|---|---|---|---:|---|---:|"]
for probe in W.rag_ground_probes():
    state = (f"Document excerpt:\n{docs[probe['doc_id']]['text']}\n\nAnswer:\n{probe['answer']}")
    dh = haiku.decide(state_text=state, question=W.RAG_GROUND_QUESTION,
                      choices=W.RAG_GROUND_CHOICES, decision_id="rag_ground")
    ds = strands.decide(state_text=state, question=W.RAG_GROUND_QUESTION,
                       choices=W.RAG_GROUND_CHOICES, decision_id="rag_ground")
    rows.append(f"| {probe['doc_id']} | {probe['gold']} | {dh.label} | {dh.latency_ms:,.0f} "
                f"| {ds.label} ({ds.confidence:.2f}) | {ds.latency_ms:,.0f} |")
display(Markdown("\n".join(rows)))

| source doc | gold | Haiku | ms | StrandsDecider (conf) | ms |
|---|---|---|---:|---|---:|
| fx_fee | yes | yes | 408 | yes (0.84) | 111 |
| atm_abroad | yes | yes | 538 | yes (0.83) | 111 |
| balance_cert | no | no | 728 | no (0.91) | 110 |
| deposit_cancel | no | no | 891 | no (0.94) | 109 |

## 5. End-to-end — the pipeline with all three gates on

Now we push all 10 queries through `G1 gate -> (retrieve -> G2 filter -> Sonnet generation) -> G3 groundedness`. The only difference between the two arms is the **judgment backend** (same retriever, same generation model, same prompts).

In [6]:
def run_rag(decider, arm_name, top_k=4):
    docs = {p["id"]: p for p in W.rag_passages()}
    n_j, judge_ms, gen_ms, gate_ok, ground_n = 0, 0.0, 0.0, 0, 0
    t0 = time.perf_counter()
    for item in W.rag_queries():
        # G1: retrieval gate
        d = decider.decide(state_text=item["q"], question=W.RAG_GATE_QUESTION,
                           choices=W.RAG_GATE_CHOICES, decision_id="rag_gate")
        n_j += 1; judge_ms += d.latency_ms
        gate_ok += (d.label == item["gate_gold"])
        kept = []
        if d.label == "retrieve":
            # G2: per-chunk filter
            for p, _ in W.rag_retrieve(item["q"], k=top_k):
                f = decider.decide(state_text=f"[{p['title']}] {p['text']}",
                                   question=W.RAG_FILTER_QUESTION.format(q=item["q"]),
                                   choices=W.RAG_FILTER_CHOICES, decision_id="rag_filter")
                n_j += 1; judge_ms += f.latency_ms
                if f.label == "yes":
                    kept.append(p)
        ctx = "\n\n".join(f"[{p['title']}] {p['text']}" for p in kept)
        user = f"Document excerpts:\n{ctx}\n\nCustomer question: {item['q']}" if kept else item["q"]
        r = sonnet.generate(system=W.RAG_GEN_SYSTEM, user=user, max_tokens=300)
        gen_ms += r.latency_ms
        verdict = "-"
        if kept:
            # G3: groundedness gate
            g = decider.decide(state_text=f"Document excerpt:\n{ctx}\n\nAnswer:\n{r.text}",
                               question=W.RAG_GROUND_QUESTION,
                               choices=W.RAG_GROUND_CHOICES, decision_id="rag_ground")
            n_j += 1; judge_ms += g.latency_ms
            verdict = g.label
            ground_n += (g.label == "yes")
        gold_kept = "o" if (item["gold_id"] in {p["id"] for p in kept}) else ("-" if not item["gold_id"] else "X")
        print(f"  [{d.label:<8s}] {len(kept)} chunks (gold {gold_kept}) ground={verdict:<3s} | {item['q'][:44]}")
    e2e = (time.perf_counter() - t0) * 1000.0
    print(f"  >>> [{arm_name}] {n_j} judgments {judge_ms:,.0f}ms | generation {gen_ms:,.0f}ms | e2e {e2e:,.0f}ms")
    return {"arm": arm_name, "n_queries": len(W.rag_queries()), "n_judgments": n_j,
            "judge_ms": judge_ms, "gen_ms": gen_ms, "e2e_ms": e2e,
            "gate_correct": gate_ok, "ground_pass": ground_n}

In [7]:
print("[Haiku arm]")
res_haiku = run_rag(haiku, "Haiku")

[Haiku arm]


  [retrieve] 1 chunks (gold o) ground=yes | Which fee option should I choose so I pay al


  [retrieve] 1 chunks (gold o) ground=yes | How much of an FX-rate discount do I get whe


  [retrieve] 1 chunks (gold o) ground=yes | I lost my card. How do I report it lost?


  [retrieve] 1 chunks (gold o) ground=yes | How much cash can I withdraw per day from AT


  [retrieve] 1 chunks (gold o) ground=yes | Where can I get a balance certificate issued


  [retrieve] 1 chunks (gold o) ground=yes | What happens to the interest if I close my s


  [retrieve] 1 chunks (gold o) ground=yes | Until when can I file a dispute against an o


  [direct  ] 0 chunks (gold -) ground=-   | Hello, is anyone available to chat right now


  [direct  ] 0 chunks (gold -) ground=-   | Thanks for the quick answer. Have a great da


  [direct  ] 0 chunks (gold -) ground=-   | Okay, understood. Please go ahead as you des
  >>> [Haiku] 45 judgments 23,678ms | generation 15,058ms | e2e 38,743ms


In [8]:
print("[StrandsDecider arm]")
res_strands = run_rag(strands, "StrandsDecider")

[StrandsDecider arm]


  [retrieve] 1 chunks (gold o) ground=yes | Which fee option should I choose so I pay al


  [retrieve] 1 chunks (gold o) ground=yes | How much of an FX-rate discount do I get whe


  [retrieve] 1 chunks (gold o) ground=yes | I lost my card. How do I report it lost?


  [retrieve] 1 chunks (gold o) ground=yes | How much cash can I withdraw per day from AT


  [retrieve] 1 chunks (gold o) ground=yes | Where can I get a balance certificate issued


  [retrieve] 1 chunks (gold o) ground=yes | What happens to the interest if I close my s


  [retrieve] 1 chunks (gold o) ground=yes | Until when can I file a dispute against an o


  [direct  ] 0 chunks (gold -) ground=-   | Hello, is anyone available to chat right now


  [direct  ] 0 chunks (gold -) ground=-   | Thanks for the quick answer. Have a great da


  [direct  ] 0 chunks (gold -) ground=-   | Okay, understood. Please go ahead as you des
  >>> [StrandsDecider] 45 judgments 4,857ms | generation 14,725ms | e2e 19,591ms


In [9]:
summary = W.rag_report(res_haiku, res_strands)

| metric | Haiku | StrandsDecider | ratio(Haiku/StrandsDecider) |
|---|---:|---:|---:|
| queries | 10 | 10 | |
| judgments (gate+filter+ground) | 45 | 45 | |
| **judgment latency total** | **23,678ms** | **4,857ms** | **4.9x** |
| generation (Sonnet) latency | 15,058ms | 14,725ms | **1.0x** |
| e2e | 38,743ms | 19,591ms | **2.0x** |

**Judgment overhead per query: 2.37s -> 0.49s (4.9x)** — generation stays on Sonnet in both arms, so the e2e gap is the judgment share.

### Observations

- **The e2e gap IS the judgment share**: generation is Sonnet in both arms, so the time that opened up between them is pure judgment overhead. LLM judgments accumulate fastest at points called once per chunk, like the filter.
- **Gate quality is on par**: on this small sample the two backends mostly agree. The large-scale benchmark (KorQuAD, 300 queries) found the same — retrieval gate 0.950 vs 0.940, groundedness 0.957 vs 0.957. This is not a speed-for-quality trade.
- **Probability as an operating knob**: raise the p(yes) threshold in section 3 and the filter gets stricter (more token savings, more risk of dropping evidence); lower it and it loosens. The LLM's yes/no text has no such knob.
- **Division of labor**: the rerank (ranking) slot belongs to a dedicated cross-encoder. The decision model's place is the three judgments; the recommended architecture is hybrid — reranker for ranking, decision model for judgments, frontier model for generation.
- **Thresholds do not transfer between models**: p(yes) distributions differ per model. If you swap the decision model, recalibrate the threshold on a labeled holdout.

### Try it yourself

- Raise the filter threshold in section 3 from 0.5 to 0.9 — how do the kept-chunk count and gold retention change?
- Widen retrieval with `run_rag(..., top_k=8)`. The Haiku arm's judgment overhead grows with top_k; the StrandsDecider arm's barely moves.
- Write your own fabricated answer in the `W.rag_ground_probes()` format and check whether G3 catches it.
- Reword the direct criterion in the G1 rubric (`W.RAG_GATE_QUESTION`) — which backend is more sensitive to the wording change?
